# 04_ML_Models.ipynb
## FakeJobShield: Machine Learning Models Benchmark
This notebook splits the dataset into stratified training and testing sets, performs cross-validation, trains 6 different machine learning models (Logistic Regression, Naive Bayes, Decision Tree, Random Forest, LightGBM, XGBoost) on TF-IDF + structured features, evaluates them on standard classification metrics, and automatically saves the comparison.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, classification_report
)
import json
import os

# Adjust working directory if run from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sns.set_theme(style="whitegrid")


In [ ]:
# Load the cleaned dataset and reconstruct feature matrix
df = pd.read_csv("data/cleaned_fake_job_postings.csv")
df["fraudulent_int"] = df["fraudulent"].map({'t': 1, 'f': 0, '1': 1, '0': 0, 1: 1, 0: 0}).fillna(0).astype(int)
y = df["fraudulent_int"].values

# Load Vectorizer and Label Encoders
vectorizer = joblib.load("models/tfidf.pkl")
label_encoders = joblib.load("models/label_encoders.pkl")

# Reconstruct final feature matrix
x_tfidf = vectorizer.transform(df["cleaned_text"].fillna(""))
for col in ["telecommuting", "has_company_logo", "has_questions"]:
    df[col] = df[col].map({'t': 1, 'f': 0, '1': 1, '0': 0, 1: 1, 0: 0, True: 1, False: 0}).fillna(0).astype(int)
x_binary = df[["telecommuting", "has_company_logo", "has_questions"]].values

encoded_cats = []
for col in ["employment_type", "required_experience", "required_education", "industry", "function"]:
    df[col] = df[col].fillna("missing").astype(str)
    le = label_encoders[col]
    df[col + "_encoded"] = le.transform(df[col])
    encoded_cats.append(df[col + "_encoded"].values.reshape(-1, 1))
x_categorical = np.hstack(encoded_cats)

x_final = hstack([x_tfidf, csr_matrix(x_binary), csr_matrix(x_categorical)]).tocsr()
print("Final Combined Feature Matrix shape:", x_final.shape)


In [ ]:
# Stratified Train-Test Split (85% Train, 15% Test)
x_train, x_test, y_train, y_test = train_test_split(
    x_final, y, test_size=0.15, stratify=y, random_state=42
)
print("Train set size:", x_train.shape[0])
print("Test set size:", x_test.shape[0])


In [ ]:
# Initialize models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost": XGBClassifier(scale_pos_weight=20, eval_metric="logloss", random_state=42, n_jobs=-1)
}


In [ ]:
# Train and evaluate models
results = {}
plt.figure(figsize=(10, 8))

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(x_train, y_train)
    
    # Predict
    preds = model.predict(x_test)
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(x_test)[:, 1]
    else:
        probs = preds
        
    # Compute metrics
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    auc = roc_auc_score(y_test, probs)
    
    results[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "ROC AUC": auc
    }
    
    print(f"  F1: {f1:.4f} | ROC AUC: {auc:.4f}")
    
    # ROC Curves
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("results/ml_roc_comparison.png", dpi=150)
plt.show()


In [ ]:
# Convert results to DataFrame
comparison_df = pd.DataFrame(results).T
print("Model Comparison Table:")
print(comparison_df)

# Save results
comparison_df.to_csv("results/ml_model_comparison.csv")
with open("results/ml_model_metrics.json", "w") as f:
    json.dump(results, f, indent=2)


In [ ]:
# Plot confusion matrix of best baseline model
best_model_name = comparison_df["F1 Score"].idxmax()
print(f"Best model determined by F1 Score: {best_model_name}")

best_model = models[best_model_name]
test_preds = best_model.predict(x_test)
cm = confusion_matrix(y_test, test_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Genuine", "Fraudulent"], yticklabels=["Genuine", "Fraudulent"])
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig(f"results/best_baseline_confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
# Save best baseline model
joblib.dump(best_model, f"models/best_baseline_{best_model_name.lower().replace(' ', '_')}.pkl")
print("Saved best baseline model.")
